# 8차시 · AI는 공정할까? (데이터 편향과 공정성)

**탐구 질문**: 정확도가 높은 AI는 공정한 AI일까? 민감한 정보(그룹)를 빼면 편향이 사라질까?

가상의 지원자 2000명: 진짜 실력은 A·B 그룹이 똑같지만, AI가 배우는 '정답'은 편견이 섞일 수 있는 **과거 선발 기록**입니다.

> 🔧 **규칙**: 맨 위 대문자 변수만, 한 번에 **하나씩** 바꾸고, 실행 **전에** 결과를 예측해 적어 보세요.

## 코드 실험실
아래 셀을 실행(Shift+Enter)해 보세요.

In [ ]:
# 8차시 · AI는 공정할까? (과거 기록으로 배운 장학생 선발 AI)
# 🔧 대문자 변수만 바꿔 가며 실험해 보세요. 한 번에 '하나만'!
HISTORY_BIAS = 1.0  # 과거 심사의 편견: B그룹 학생을 얼마나 깎아서 뽑았나 (0 = 편견 없음)
USE_GROUP = True    # AI에게 '그룹(A/B)' 정보를 입력으로 줄까?
PROXY = 0.9         # '사는 동네'가 그룹과 얼마나 겹치나 (0.5 = 상관없음, 1.0 = 완전히 같음)
THRESHOLD = 0.5     # AI 확률이 이 값 이상이면 '선발'
SEED = 0

import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(SEED)

# ① 가상의 지원자 2000명 만들기 (A, B 그룹의 실력 분포는 똑같아요!)
N = 2000
group = (rng.random(N) < 0.3).astype(int)              # 0 = A그룹(70%), 1 = B그룹(30%)
skill = rng.normal(0, 1, N)                             # 진짜 실력 (AI는 못 봐요)
score = skill + rng.normal(0, 0.5, N)                   # 시험 점수 (실력 + 운)
region = np.where(rng.random(N) < PROXY, group, 1 - group)   # 사는 동네 (0 = 가동, 1 = 나동)
qualified = skill > 0                                   # 실제로 자격이 있는 학생

# ② 과거 사람 심사위원의 선발 기록 = AI가 배울 '정답'
past = (skill + rng.normal(0, 0.5, N) - HISTORY_BIAS * group) > 0

# ③ AI가 볼 수 있는 정보 (입력)
cols = [score, region] + ([group] if USE_GROUP else [])
names = ["시험점수", "사는동네"] + (["그룹"] if USE_GROUP else [])
X = np.column_stack(cols)
train, test = np.arange(1400), np.arange(1400, N)

# ④ 로지스틱 회귀를 경사하강법으로 학습 (1차시와 같은 원리!)
w, b = np.zeros(X.shape[1]), 0.0
for _ in range(2000):
    p = 1 / (1 + np.exp(-(X[train] @ w + b)))           # 선발 확률
    err = p - past[train]
    w -= 0.1 * X[train].T @ err / len(train)
    b -= 0.1 * err.mean()

prob = 1 / (1 + np.exp(-(X @ w + b)))
pick = prob >= THRESHOLD

# ⑤ 결과 보기 (학습에 안 쓴 시험용 600명)
print("[AI가 배운 가중치]  (+ 클수록 선발에 유리, − 클수록 불리)")
for n, v in zip(names, w):
    print(f"  {n:<6} {v:+.2f}")

t = test
acc = (pick[t] == past[t]).mean()
print(f"\n과거 기록과 일치하는 비율(정확도): {acc * 100:.0f}%")
print("\n           선발률   자격 있는데 탈락한 비율")
sel_rates, miss_rates = [], []
for g, gname in [(0, "A그룹"), (1, "B그룹")]:
    m = t[group[t] == g]                  # 시험용 중 이 그룹 학생들
    sel = pick[m].mean()                  # 선발된 비율
    q = m[qualified[m]]                   # 그중 진짜 자격이 있는 학생들
    miss = (~pick[q]).mean()              # 자격이 있는데 떨어진 비율
    sel_rates.append(sel * 100); miss_rates.append(miss * 100)
    print(f"  {gname}    {sel * 100:5.1f}%        {miss * 100:5.1f}%   ({len(m)}명)")

x = np.arange(2)
plt.figure(figsize=(6, 3))
plt.bar(x - 0.2, sel_rates, 0.4, label="selected (%)")
plt.bar(x + 0.2, miss_rates, 0.4, label="qualified but rejected (%)")
plt.xticks(x, ["Group A", "Group B"]); plt.ylim(0, 100); plt.legend(); plt.grid(axis="y", alpha=.3)
plt.title(f"HISTORY_BIAS={HISTORY_BIAS}, USE_GROUP={USE_GROUP}, PROXY={PROXY}"); plt.tight_layout(); plt.show()

gap = abs(pick[t][group[t] == 0].mean() - pick[t][group[t] == 1].mean())
print(f"\n선발률 차이(A − B): {gap * 100:.1f}%p  ·  정확도 {acc * 100:.0f}%")

## 탐구 미션
각 미션마다 위 셀의 변수를 바꿔 다시 실행하고, 이 칸을 더블클릭해 예측·결과·설명을 적어 보세요.

### M1. 정확도 ≠ 공정성
`HISTORY_BIAS`를 `1.0 → 0.0`

- **예측**: 
- **결과**: 
- **설명**: 

### M2. 그룹 정보를 지우면 해결될까?
`HISTORY_BIAS = 1.0`으로 돌리고 `USE_GROUP`을 `True → False`

- **예측**: 
- **결과**: 
- **설명**: 

### M3. 대리 변수의 힘
`USE_GROUP = False` 그대로 `PROXY`를 `0.9 → 0.7 → 0.5`

- **예측**: 
- **결과**: 
- **설명**: 

### M4. 코드 수정: 그룹별 기준선
`pick = prob >= THRESHOLD` 줄을 고쳐 B그룹에는 다른 기준 `THRESHOLD_B`를 적용해 보세요. 선발률 차이를 10%p 안으로 줄이려면 THRESHOLD_B는 얼마여야 할까?

- **예측**: 
- **결과**: 
- **설명**: 

## 나의 결론
오늘의 탐구 질문에 대한 나의 결론 (3문장 이상):